# NoCSA baseline — exact10XE, Karpathy TRAIN/VALID
Use account with30h available. GPU T4x2, InternetON, SaveVersion/RunAll. Add fullROI134 andsemanticbanks130 Datasets; paths prefilled.
Fresh baseline: same projection/decoder/head,3ordinary visualTransformer layers, SAME outputmemory to both decoderports. No banksemantics inforward. Banks input only validates originalassets and disposableDDPpreflight. No pretrainedCSAweights.
Same10epochs/global32/FP32/Adam/LR/warmup/order/beam5/max20 as historicalCSA10. Common initial layers match seed42 exactly; parametercount differs. Actual-input disposableplain smoke+6DDPguards beforetraining. Full5000VALID each epoch; TESTnever used here.
PRIMARY endpoint comparison uses EXACT epoch10 checkpoint.bin, not validationbest. Keep bothcheckpoint.bin andbest.bin withreceipts/fulloutput. Later evaluate baseline10 on the reusableTESTcache produced by notebook2; noTESTcache is required to starttraining.
10.5h elapsed stopcap, notETA/billing. No SCST ornovelalignment. Send plain_full_xe_10_review.zip. Do not deleteoutputweights.


In [ ]:
from pathlib import Path

FULL_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-full-roi-karpathy-v134')
BANK_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-semantic-banks-v130')
RESUME_ROOT='' # Fresh baseline. Only set to a COMPLETE saved baseline output folder after interruption.
OUTPUT=Path('/kaggle/working/plain_full_xe_10')
INPUT_ROOTS=[FULL_ROOT,BANK_ROOT];ARM='plain';TEST_USED=False

import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil
from pathlib import Path
COMMIT='63e11dbe9a17ef035ad72d7d404757ec03432711'
HASHES={'run_sharing_benchmark.py': '51706ae5cbd96fb0484cb0487a05f14f1b34f4332bd9cb57a299d7d3d5a14d18', 'sharing_data.py': '68a065f5c055a55b41eae0ea554fbb7f8fd1124e8f5422679fcb1d4878f14829', 'full_cache_data.py': '58cf4d08310fe03e01441af2ddeb12fd3404f59e82259194a32fe98c8b002bc1', 'model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'pilot_data.py': '0edab33c7ad7cc54c28efdf6007f23fa6dc4547bdda32e5c0d7d888a3369394d', 'package_full_roi_dataset.py': 'be3e7d0f96add450a5295add6466e19977da90f486e5e726e3ea40a505cceecb', 'SAVE_FULL_ROI_V134_PACKAGE.json': 'a73636ca75806d6b1d7b4d88f80f3ce1c59b3061eb3e9a5e730097ddd89c505c', 'SAVE_BANK_V130_PACKAGE.json': '826ba36628d6587860e2d548e950ea953bd4bbb8c0a8dc64393089dbc1e058e1', 'REAL_BANK_INPUT_PACKAGE.json': 'c60902e04ea15cf90f652e384937bb317e288753f13b8ede6ab5ef89f8d97129', 'lemma_correction_review_text/bank_words_corrected.json': 'cec76a0229f30361235a4b64e9712ac6bb2e79fc62a9241079a7af3fa95d13ac', 'DDP_SHARING_BENCHMARK_PROTOCOL.md': '6537e5e44fb8eb1504c4234b87d4e4e87929cf80fed3884f3caa1ef09468b2f1', 'run_full_xe.py': 'e3a56c95963d64d1f00e4b21aa18eddd0f2a834d58556136996c7a2323e56101', 'full_xe_data.py': 'a7a5a3b4d70c9d3a357aa464dfe397a6cba8d851e33065299fb98721ddca7135', 'full_xe_decode.py': 'b8e618559feec6361cfb3b4619899e4968ab71ab67e113ff38b6d3cd99cdc235', 'test_full_xe.py': 'aee46db054fd18bcf9306d8d8240414d67ca39dccb74d7ab0337e5d9ef7cb1fa', 'FULL_XE_10_PROTOCOL.md': 'b52d4081cf0657c7c7fba8a00def7efbff928b4979180e08b4e552cc3983a70c', 'plain_model.py': 'dcb90639bd7943bea9baffca5edced2846b9e3f68d2f999491870059b7faf2de', 'run_plain_smoke.py': 'e8c2125223815787a6970d1a8636e46da0ffe5d1c4c3a5ee81fe8dcfea2fa61b', 'prepare_test_regions.py': '3c61ed502f3091293c537a32ec2c0f82c72f4bf97cc86cfabe4b035c365692b7', 'run_csa10_test.py': '2bc56233bf64ad70f1174d76fb61ff932ec6e7b3f91b9d71d5f963cc71ce8db0', 'test_comparison_setup.py': '57402ad74206efe792d9b3fdf0f01916948bb25ba7f3773ae68ed33f3f14a4be', 'make_comparison_notebooks.py': '9c49a556c01538607aef76d293a81cfed94aa27aab28c57fed0b71871d164f4d', 'PLAIN_VS_CSA10_PROTOCOL.md': '11eb4fbac432fb2988aa90c68bf9c6c56dd96b388c18a1c2e1e9ce8cb8449d4e', 'prepare_real_assets.py': '44a22d92acd239bdc7a1edbe30984daf0853ac8bc196969dd48b19b78df1cc7b', 'audit_assets.py': '5be2fdf1ee094ef8b23a547b16f423ed3603f7c80b2fe5f2d46cb9224cb90959', 'data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'run_best9_decode.py': 'fdc6b027ac13d510be72aa25a963eb89e5467e09f179bfaf0756469aef77a648', 'saved_quality_data.py': 'f8a092506e3adf96292203190ea2f83de397d3028a3ddc6293d58fd360199a1e', 'best9_saved_text.json.gz': '42594e90ec2f47646e7e68347a17471213038bc182814ca9d29ea374e8010861', 'BEST9_SAVED_TEXT_PACKAGE.json': '898d67c3eead66c89f5d2c091b24e5e3eb44d884e86550b2b26f45d6708032e5', 'SAVE_XE139_PACKAGE.json': '452ee35118a3d0285bc7f522bb8c8fc7a0e3e0dac70cac9c2333e7583cd4e423', 'full_roi_review_text/input_receipt.json': '18e9481863418c7f2f4c538fa98f3a05377e5b81422cb249d33f81ddefd78f7f'}
if OUTPUT.exists():raise RuntimeError('Preserve existing output; use a fresh session/output path.')
if not all(p.is_dir() for p in INPUT_ROOTS):raise FileNotFoundError('Add the required Datasets and check paths before running.')
OUTPUT.mkdir(parents=True);SOURCE=Path(tempfile.mkdtemp(prefix='csa10-comparison-'));LOG=OUTPUT/'notebook.log';STAGE=SOURCE/'stage'
os.environ['OMP_NUM_THREADS']='1'
for n,h in HASHES.items():
    raw=urllib.request.urlopen(f'https://raw.githubusercontent.com/Supzxjee/Image_Captioning/{COMMIT}/experiments/csa_paper/{n}',timeout=90).read()
    if hashlib.sha256(raw).hexdigest()!=h:raise ValueError('Changed source:'+n)
    p=SOURCE/n;p.parent.mkdir(parents=True,exist_ok=True);p.write_bytes(raw)
(OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=COMMIT,sha256=HASHES),indent=2),encoding='utf-8')


In [ ]:
def run(args, timeout=1800):
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('Worker active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('Phase cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False;deadline=time.monotonic()+10.5*3600
common=['--bank-root',BANK_ROOT,'--stage',STAGE,'--output',OUTPUT,'--arm','plain','--target-epochs','10']
try:
    if shutil.disk_usage('/tmp').free<6*1024**3:raise RuntimeError('Need6GiB free temporary disk')
    run([sys.executable,'-u',SOURCE/'test_full_xe.py'],timeout=120)
    run([sys.executable,'-u',SOURCE/'test_comparison_setup.py'],timeout=120)
    run([sys.executable,'-m','pip','install','-q','pycocoevalcap==1.2'],timeout=300)
    run([sys.executable,'-u',SOURCE/'run_full_xe.py','prepare','--full-root',FULL_ROOT,*common],timeout=900)
    run([sys.executable,'-u',SOURCE/'run_full_xe.py','metric_preflight',*common],timeout=300)
    run([sys.executable,'-u',SOURCE/'run_plain_smoke.py','--stage',STAGE,'--output',OUTPUT],timeout=180)
    launch=[sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0']
    preflight=OUTPUT/'preflight';preflight.mkdir()
    run([*launch,SOURCE/'run_sharing_benchmark.py','worker','--bank-root',BANK_ROOT,'--stage',STAGE,'--output',preflight],timeout=600)
    if json.loads((preflight/'result.json').read_text(encoding='utf-8'))['preflight_cases']!=6:raise ValueError('Missing DDP guards')
    remaining=deadline-time.monotonic()
    if remaining<600:raise TimeoutError('No training margin after setup')
    run([*launch,SOURCE/'run_full_xe.py','worker',*common,'--resume-root',RESUME_ROOT,'--wall-seconds',remaining-60],timeout=remaining)
    result=OUTPUT/'result.json'
    complete=result.is_file() and json.loads(result.read_text(encoding='utf-8'))['status']=='PLAIN_XE_10_COMPLETE'
    if not complete:raise RuntimeError('Training stopped early. Preserve full output to resume.')
    print('PRIMARY comparison uses checkpoint.bin EXACT epoch10, not best.bin.',flush=True)

except BaseException:
    (OUTPUT/'failure.txt').write_text(traceback.format_exc(),encoding='utf-8');raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,arm=ARM,test_used=TEST_USED)),encoding='utf-8')
    archive=Path(str(OUTPUT)+'_review.zip')
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUTPUT.rglob('*')):
            if p.is_file() and p.suffix in ('.json','.txt','.log'):z.write(p,str(p.relative_to(OUTPUT)))
    print('SEND REVIEW:',archive,flush=True)
    print('KEEP ENTIRE OUTPUT:',OUTPUT,flush=True)
    print('COMPLETE' if complete else 'INCOMPLETE: preserve logs/progress, do not restart automatically.',flush=True)
